[![cloudevel](../img/cloudevel.png)](https://cloudevel.com)

# Solución: runner alojado en infraestructura propia

**Cómo leer este cuaderno.** Cada ejercicio reproduce el enunciado del cuaderno original y presenta su solución.

* Las celdas **sin etiqueta** se ejecutan en cualquier equipo, sin credenciales ni cuenta de *GitHub*. Escriben solo en carpetas temporales.
* Las celdas con la etiqueta `skip-execution` necesitan una cuenta de *GitHub*, un proyecto de nube o un servidor; la integración continua (*CI*) no las ejecuta. Debajo de cada una se describe la salida esperada.
* Ningún valor real aparece en el código: los datos de cada persona se leen de variables de entorno o se sustituyen por los marcadores indicados.

**Advertencia de seguridad.** Un *runner* autoalojado **solo** se registra en un repositorio **privado**. En uno público, un *pull request* ajeno podría ejecutar código en la máquina del *runner*. Las celdas con etiqueta de esta solución comprueban esa condición antes de registrarlo.

Este cuaderno resuelve la actividad de `10_runner_en_entorno_local.ipynb`: registrar un *runner* en el *Codespace*, ejecutar en él un *workflow* y darlo de baja.

La siguiente celda importará el módulo `utilidades` (que está en esta misma carpeta) y localizará la raíz del repositorio. Todas las comprobaciones locales de este cuaderno leen archivos desde esa raíz.

In [ ]:
import utilidades as u

RAIZ = u.raiz_repositorio()
RAIZ.name

## Ejercicio 1. Preparar el repositorio de la práctica como privado (requisito de seguridad)

Complemento no incluido en el cuaderno original. El cuaderno 02 recomienda un repositorio **público**, pero el cuaderno 10 exige que un *runner* autoalojado solo se use en un repositorio **privado**. Antes de registrar el *runner*, el repositorio `demo-github-actions` debe ser privado.

La siguiente celda cambiará la visibilidad del repositorio `demo-github-actions` a privada y comprobará el resultado. Cambiar la visibilidad es una decisión de la persona propietaria; por eso `gh` pide aceptar explícitamente las consecuencias.

In [ ]:
%%bash
set -euo pipefail
cd "${HOME}/demo-github-actions"
gh repo edit --visibility private --accept-visibility-change-consequences
gh repo view --json visibility --jq .visibility

**Salida esperada.** `PRIVATE`. Las ejecuciones de *Actions* en repositorios privados consumen la cuota de minutos de la cuenta; los *runners* autoalojados no.

## Ejercicio 2. Descargar la aplicación del runner

Crear el subdirectorio `actions-runner` y descargar en él el paquete del *runner* para *Linux* y arquitectura `x64`. La versión y la dirección deben coincidir con las que *GitHub* muestra en `Settings` → `Actions` → `Runners` → `New self-hosted runner`.

El nombre del paquete y su dirección siguen un patrón fijo que depende solo de la versión. La siguiente celda lo construye y lo compara con la dirección del cuaderno original. Si *GitHub* muestra otra versión, se cambia solo `VERSION`.

La siguiente celda calculará el nombre del paquete y su dirección a partir de la versión y comprobará que coinciden con las del cuaderno original (versión `2.332.0`).

In [ ]:
VERSION = "2.332.0"
PAQUETE = f"actions-runner-linux-x64-{VERSION}.tar.gz"
URL = f"https://github.com/actions/runner/releases/download/v{VERSION}/{PAQUETE}"
print(URL)
assert URL == "https://github.com/actions/runner/releases/download/v2.332.0/actions-runner-linux-x64-2.332.0.tar.gz"

La siguiente celda creará el directorio `actions-runner` en la carpeta personal (`$HOME`), fuera de cualquier repositorio, y descargará el paquete con `curl`. La variable `RUNNER_VERSION` debe coincidir con la de *GitHub*.

In [ ]:
%%bash
set -euo pipefail
RUNNER_VERSION="${RUNNER_VERSION:-2.332.0}"
PAQUETE="actions-runner-linux-x64-${RUNNER_VERSION}.tar.gz"
mkdir -p "${HOME}/actions-runner"
cd "${HOME}/actions-runner"
curl --fail --location --output "${PAQUETE}" \
  "https://github.com/actions/runner/releases/download/v${RUNNER_VERSION}/${PAQUETE}"
ls -l "${PAQUETE}"

**Salida esperada.** `ls -l` muestra el archivo `actions-runner-linux-x64-<versión>.tar.gz` con un tamaño de decenas de megabytes.

## Ejercicio 3. Verificar la integridad del paquete

Comprobar el *hash* *SHA256* del paquete con el valor que muestran las instrucciones de *GitHub*. La celda del cuaderno original solo contiene comentarios; no ejecuta ninguna comprobación.

El comando `sha256sum -c` lee líneas con el formato `<hash>␣␣<archivo>` y devuelve un código distinto de cero si algún archivo no coincide. La siguiente celda ensaya el mecanismo con un archivo de prueba, porque el valor real lo da *GitHub* y cambia con la versión.

La siguiente celda creará un archivo de prueba, calculará su *hash* y lo verificará con `sha256sum -c`. Después alterará el archivo y comprobará que la verificación falla.

In [ ]:
import hashlib
import subprocess

TEMPORAL = u.directorio_temporal()
archivo = TEMPORAL / "paquete-de-prueba.tar.gz"
archivo.write_bytes(b"contenido de prueba")
huella = hashlib.sha256(archivo.read_bytes()).hexdigest()

def verificar(huella_esperada):
    return subprocess.run(
        ["sha256sum", "--check", "--status"],
        input=f"{huella_esperada}  {archivo.name}\n",
        cwd=TEMPORAL, text=True, check=False,
    ).returncode

print("Con el hash correcto :", verificar(huella))
archivo.write_bytes(b"contenido alterado")
print("Con el archivo alterado:", verificar(huella))
archivo.write_bytes(b"contenido de prueba")
assert verificar(huella) == 0
archivo.write_bytes(b"contenido alterado")
assert verificar(huella) != 0

La siguiente celda verificará el paquete real. El valor del *hash* se toma de la variable de entorno `RUNNER_SHA256`, que la persona copia de las instrucciones de *GitHub*; si no está definida, la celda se detiene con un mensaje.

In [ ]:
%%bash
set -euo pipefail
: "${RUNNER_SHA256:?Copie en RUNNER_SHA256 el hash que muestra GitHub}"
RUNNER_VERSION="${RUNNER_VERSION:-2.332.0}"
cd "${HOME}/actions-runner"
echo "${RUNNER_SHA256}  actions-runner-linux-x64-${RUNNER_VERSION}.tar.gz" | sha256sum --check

**Salida esperada.** `actions-runner-linux-x64-<versión>.tar.gz: OK`. Con un *hash* incorrecto, `sha256sum` imprime `FAILED` y termina con código 1; en ese caso no se debe extraer el paquete.

## Ejercicio 4. Registrar y poner en marcha el runner

Extraer el paquete, ejecutar el *script* de configuración con el *token* que muestra *GitHub* (aceptando las opciones por defecto) y ejecutar `./run.sh` desde la misma terminal. Si la configuración es correcta, la aplicación se conecta a *GitHub* y espera *jobs*.

La siguiente celda comprobará que el repositorio es privado y obtendrá un *token* de registro con la *API* de *GitHub*. El *token* se guarda en un archivo temporal con permisos `600` y **nunca se imprime**; caduca en una hora.

In [ ]:
%%bash
set -euo pipefail
cd "${HOME}/demo-github-actions"
ES_PRIVADO="$(gh repo view --json isPrivate --jq .isPrivate)"
if [ "${ES_PRIVADO}" != "true" ]; then
  echo "El repositorio es público: no se registra un runner autoalojado." >&2
  exit 1
fi
REPO="$(gh repo view --json nameWithOwner --jq .nameWithOwner)"
umask 077
gh api --method POST "repos/${REPO}/actions/runners/registration-token" --jq .token > "${TMPDIR:-/tmp}/runner-token"
echo "Token guardado (no se muestra). Repositorio: ${REPO}"

La siguiente celda extraerá el paquete y configurará el *runner* de forma no interactiva (`--unattended` equivale a aceptar las opciones por defecto). `--ephemeral` hace que el *runner* se dé de baja solo después de un *job*. Al terminar borra el archivo del *token*.

In [ ]:
%%bash
set -euo pipefail
RUNNER_VERSION="${RUNNER_VERSION:-2.332.0}"
REPO="$(cd "${HOME}/demo-github-actions" && gh repo view --json nameWithOwner --jq .nameWithOwner)"
cd "${HOME}/actions-runner"
tar xzf "./actions-runner-linux-x64-${RUNNER_VERSION}.tar.gz"
./config.sh --unattended --ephemeral \
  --url "https://github.com/${REPO}" \
  --token "$(cat "${TMPDIR:-/tmp}/runner-token")" \
  --labels self-hosted,linux
rm -f "${TMPDIR:-/tmp}/runner-token"

**Salida esperada.** El *script* imprime el resultado del registro, `Runner successfully added`, y la configuración de `Runner settings`. En `Settings` → `Actions` → `Runners` aparece el *runner* con estado `Idle` una vez que se ejecuta el siguiente comando.

La siguiente celda pondrá en marcha el *runner* en segundo plano con `nohup`, escribirá su bitácora en `actions-runner/runner.log` y mostrará sus primeras líneas. El cuaderno original lo ejecuta en una terminal; en segundo plano el cuaderno no queda bloqueado.

In [ ]:
%%bash
set -euo pipefail
cd "${HOME}/actions-runner"
nohup ./run.sh > runner.log 2>&1 &
echo $! > runner.pid
sleep 8
head -n 10 runner.log

**Salida esperada.** La bitácora incluye `Connected to GitHub` y `Listening for Jobs`.

## Ejercicio 5. Ejecutar un workflow en el runner propio

En el repositorio de las actividades, editar `.github/workflows/expresiones.yml`, cambiar el tipo de *runner* a `runs-on: self-hosted`, hacer un *commit* y ejecutar manualmente el *workflow* `Expresiones`. La actividad debe reflejarse en la terminal del *runner*.

La siguiente celda ensayará la edición sobre un texto *YAML* de ejemplo con `re.sub` y comprobará con `cargar_workflow()` que el resultado sigue siendo válido y que solo cambió `runs-on`.

In [ ]:
import re

ANTES = """\
name: Expresiones
on:
  workflow_dispatch:
jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - run: echo hola
"""
DESPUES = re.sub(r"^(\s+runs-on:)\s*ubuntu-latest\s*$", r"\1 self-hosted", ANTES, flags=re.MULTILINE)

antes, despues = u.cargar_workflow(ANTES), u.cargar_workflow(DESPUES)
assert antes["jobs"]["build"]["runs-on"] == "ubuntu-latest"
assert despues["jobs"]["build"]["runs-on"] == "self-hosted"
despues["jobs"]["build"]["runs-on"] = "ubuntu-latest"
assert despues == antes
print(DESPUES)

La siguiente celda cambiará `runs-on` en el archivo real, enviará el cambio y lanzará el *workflow*. Después mostrará las líneas de la bitácora del *runner* donde se ejecutó el *job*.

In [ ]:
%%bash
set -euo pipefail
cd "${HOME}/demo-github-actions"
sed -i -E 's/^(\s+runs-on:)\s*ubuntu-latest\s*$/\1 self-hosted/' .github/workflows/expresiones.yml
git add .github/workflows/expresiones.yml
git commit --message "Ejecuta Expresiones en el runner autoalojado"
git push origin HEAD
gh workflow run expresiones.yml --ref main
sleep 20
grep -E "Running job|Job .* completed" "${HOME}/actions-runner/runner.log"

**Salida esperada.** En `runner.log` aparecen `Running job: build` y `Job build completed with result: Succeeded`. En la ejecución de *GitHub*, el *job* indica que se ejecutó en el *runner* con las etiquetas `self-hosted` y `Linux`.

## Ejercicio 6. Dar de baja el runner (limpieza)

Complemento no incluido en el cuaderno original: al terminar la práctica, el *runner* no debe quedar registrado. Esta es la regla de seguridad que cierra el cuaderno 10.

La siguiente celda detendrá el proceso del *runner*, lo eliminará de *GitHub* si sigue registrado (un *runner* efímero ya se dio de baja solo) y borrará la carpeta `actions-runner` de la carpeta personal.

In [ ]:
%%bash
set -uo pipefail
cd "${HOME}/actions-runner" || { echo "No existe actions-runner" >&2; exit 1; }
[ -f runner.pid ] && kill "$(cat runner.pid)" 2>/dev/null
REPO="$(cd "${HOME}/demo-github-actions" && gh repo view --json nameWithOwner --jq .nameWithOwner)"
TOKEN="$(gh api --method POST "repos/${REPO}/actions/runners/remove-token" --jq .token)"
./config.sh remove --token "${TOKEN}" || echo "El runner ya estaba dado de baja."
cd "${HOME}/demo-github-actions"
rm -rf "${HOME}/actions-runner"
gh api "repos/${REPO}/actions/runners" --jq '.total_count'
sed -i -E 's/^(\s+runs-on:)\s*self-hosted\s*$/\1 ubuntu-latest/' .github/workflows/expresiones.yml

**Salida esperada.** `Runner removed successfully` (o el aviso de que ya estaba dado de baja) y, al final, el número `0` como total de *runners* del repositorio. El último comando deja el *workflow* de nuevo con `ubuntu-latest`; ese cambio se confirma con un *commit* aparte.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023-2026.</p>